# Déploiement du service d'Alerting (Amazon SNS)

Ce notebook déploie et configure l'infrastructure d'alerting et de notification pour le projet IoT SDATAWAY (Groupe 10) :
1. Détection de la région AWS et initialisation du client **Amazon SNS** via Boto3.
2. Création idempotente du topic SNS (`GRP10-SdatawayAlertsTopic`).
3. Enregistrement optionnel des adresses email des opérateurs (`ALERT_EMAILS`).
4. Export de la configuration dans `sns_config.json` (utilisé par le Scenario Analyzer et la VM).
5. Cellule de test conditionnelle (`CONFIRM_TEST`) publiant un événement d'alerte simulé avec attributs de message.
6. Zone de nettoyage (Teardown) conditionnelle (`CONFIRM_DELETE`) pour supprimer proprement les ressources.

In [1]:
import os
import json
import time
import boto3
from botocore.exceptions import ClientError

# --- 1. CONFIGURATION ---
# Lecture de la région configurée pour le projet (depuis db_config.json si présent)
DB_CONFIG_PATH = "../DB/db_config.json"
if os.path.exists(DB_CONFIG_PATH):
    with open(DB_CONFIG_PATH, "r") as f:
        db_config = json.load(f)
    REGION = db_config.get("region", "us-east-1")
else:
    REGION = "us-east-1"

TOPIC_NAME = "GRP10-SdatawayAlertsTopic"

# Liste des adresses email des opérateurs à abonner aux alertes.
# - Laissez vide [] par défaut pour ne pas envoyer d'emails de confirmation intempestifs.
# - Ajoutez votre email pour tester (ex: ["prenom.nom@hes-so.ch"]).
#   Vous recevrez un mail AWS "Subscription Confirmation" avec un lien d'activation obligatoire.
ALERT_EMAILS = [ "basile.gasser@hes-so.ch" ]

# Initialisation de la session et des clients Boto3
session = boto3.Session(region_name=REGION)
sns_client = session.client("sns")
sts_client = session.client("sts")

try:
    ACCOUNT_ID = sts_client.get_caller_identity()["Account"]
    print(f"Initialisation Boto3 réussie sur le compte AWS : {ACCOUNT_ID} (Région : {REGION})")
except ClientError as e:
    ACCOUNT_ID = "unknown"
    print(f"Initialisation Boto3 (Région : {REGION}) - Attention credentials AWS : {e}")

Initialisation Boto3 réussie sur le compte AWS : 352909266144 (Région : us-east-1)


In [2]:
def create_or_get_topic(topic_name: str) -> str:
    """
    Crée le topic SNS s'il n'existe pas, ou récupère son ARN s'il est déjà créé (idempotent).
    """
    try:
        response = sns_client.create_topic(
            Name=topic_name,
            Attributes={
                "DisplayName": "SDATAWAY-Alerts"
            },
            Tags=[
                {"Key": "Project", "Value": "cloud-project-2-grp10"},
                {"Key": "Component", "Value": "sns-alerting"}
            ]
        )
        arn = response["TopicArn"]
        print(f"Topic SNS '{topic_name}' disponible.")
        print(f"ARN : {arn}")
        return arn
    except ClientError as e:
        print(f"Erreur lors de la création du topic SNS : {e.response['Error']['Message']}")
        raise e

topic_arn = create_or_get_topic(TOPIC_NAME)

Topic SNS 'GRP10-SdatawayAlertsTopic' disponible.
ARN : arn:aws:sns:us-east-1:352909266144:GRP10-SdatawayAlertsTopic


In [3]:
def subscribe_emails(target_topic_arn: str, emails: list[str]) -> list[str]:
    """
    Abonne chaque adresse email de la liste au topic SNS.
    AWS envoie alors un mail contenant un lien pour confirmer la souscription.
    """
    created_subs = []
    if not emails:
        print("ℹ️  Aucune adresse dans ALERT_EMAILS : aucun abonnement automatique créé.")
        print("   Vous pouvez ajouter des abonnements manuellement dans la console AWS SNS ou remplir la liste ci-dessus.")
        return created_subs

    # Récupération des abonnements existants pour éviter les doublons d'appels
    existing = sns_client.list_subscriptions_by_topic(TopicArn=target_topic_arn).get("Subscriptions", [])
    existing_endpoints = {s.get("Endpoint") for s in existing if s.get("Endpoint")}

    for email in emails:
        email = email.strip()
        if not email:
            continue
        if email in existing_endpoints:
            print(f"L'adresse '{email}' est déjà inscrite sur ce topic.")
            continue
        try:
            res = sns_client.subscribe(
                TopicArn=target_topic_arn,
                Protocol="email",
                Endpoint=email,
                ReturnSubscriptionArn=True
            )
            sub_arn = res.get("SubscriptionArn", "PendingConfirmation")
            created_subs.append(sub_arn)
            print(f"Demande d'abonnement envoyée pour '{email}'.")
            print(f"  -> Statut : {sub_arn}")
            print(f"  -> Un email de confirmation vient d'être envoyé à {email}. Cliquez sur 'Confirm subscription' pour activer.")
        except ClientError as e:
            print(f"Erreur lors de l'abonnement de {email} : {e.response['Error']['Message']}")

    return created_subs

subscriptions = subscribe_emails(topic_arn, ALERT_EMAILS)

Demande d'abonnement envoyée pour 'basile.gasser@hes-so.ch'.
  -> Statut : arn:aws:sns:us-east-1:352909266144:GRP10-SdatawayAlertsTopic:19c86a56-81f7-4979-b648-d6a93ac17411
  -> Un email de confirmation vient d'être envoyé à basile.gasser@hes-so.ch. Cliquez sur 'Confirm subscription' pour activer.


In [4]:
sns_config = {
    "region": REGION,
    "topic_name": TOPIC_NAME,
    "topic_arn": topic_arn,
    "alert_emails": ALERT_EMAILS
}

CONFIG_FILE = "sns_config.json"
with open(CONFIG_FILE, "w") as f:
    json.dump(sns_config, f, indent=2)

print(f"Fichier '{CONFIG_FILE}' généré avec succès :")
print(json.dumps(sns_config, indent=2))

Fichier 'sns_config.json' généré avec succès :
{
  "region": "us-east-1",
  "topic_name": "GRP10-SdatawayAlertsTopic",
  "topic_arn": "arn:aws:sns:us-east-1:352909266144:GRP10-SdatawayAlertsTopic",
  "alert_emails": [
    "basile.gasser@hes-so.ch"
  ]
}


In [5]:
CONFIRM_TEST = False  # Passez à True pour tester la publication d'une alerte formatée sur SNS

if CONFIRM_TEST:
    # Exemple d'événement simulant la détection d'un thermal_runaway par le Scenario Analyzer
    now_ms = int(time.time() * 1000)
    test_alert = {
        "alert_id": f"coffee-device-30:thermal_runaway:{now_ms}",
        "device_id": "coffee-device-30",
        "scenario": "thermal_runaway",
        "severity": "critical",
        "state": "open",
        "opened_at": now_ms,
        "message": "Temperature continuous rise: 114.2°C exceeds critical threshold (112.0°C)"
    }

    subject = f"[SDATAWAY ALERT - CRITICAL] thermal_runaway on coffee-device-30"
    timestamp_str = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    body = (
        f"=== SDATAWAY IOT MONITORING ALERT ===\n\n"
        f"Device ID   : {test_alert['device_id']}\n"
        f"Scénario    : {test_alert['scenario']}\n"
        f"Gravité     : {test_alert['severity'].upper()}\n"
        f"Statut      : {test_alert['state'].upper()}\n"
        f"Message     : {test_alert['message']}\n"
        f"Horodatage  : {timestamp_str}\n\n"
        f"Détails JSON :\n"
        f"{json.dumps(test_alert, indent=2)}\n\n"
        f"---\n"
        f"Notification générée par le système d'alerting Cloud (Groupe 10)\n"
    )

    # Publication avec MessageAttributes pour permettre le filtrage natif SNS
    res = sns_client.publish(
        TopicArn=topic_arn,
        Subject=subject[:100],  # Les sujets SNS sont limités à 100 caractères max
        Message=body,
        MessageAttributes={
            "severity": {"DataType": "String", "StringValue": test_alert["severity"]},
            "scenario": {"DataType": "String", "StringValue": test_alert["scenario"]},
            "device_id": {"DataType": "String", "StringValue": test_alert["device_id"]},
            "event_kind": {"DataType": "String", "StringValue": "opened"}
        }
    )
    print(f"✅ Message d'alerte de test publié avec succès !")
    print(f"   MessageId : {res.get('MessageId')}")
    print("   Si votre adresse est confirmée, vérifiez votre boîte de réception.")
else:
    print("Test désactivé. Passez CONFIRM_TEST = True pour publier un message d'alerte de test.")

Test désactivé. Passez CONFIRM_TEST = True pour publier un message d'alerte de test.


---
## 🧹 Zone de Nettoyage (Teardown)
Exécutez la cellule suivante pour supprimer les abonnements existants, le topic SNS et le fichier de configuration local.

In [6]:
CONFIRM_DELETE = False  # Passez à True pour autoriser la suppression du Topic SNS et des souscriptions

if CONFIRM_DELETE:
    # 1. Suppression des abonnements rattachés au topic
    try:
        subs = sns_client.list_subscriptions_by_topic(TopicArn=topic_arn).get("Subscriptions", [])
        for s in subs:
            arn = s.get("SubscriptionArn")
            if arn and arn != "PendingConfirmation":
                try:
                    sns_client.unsubscribe(SubscriptionArn=arn)
                    print(f"Abonnement désinscrit : {s.get('Endpoint', arn)}")
                except ClientError as e:
                    print(f"Avertissement désabonnement : {e.response['Error']['Message']}")
    except ClientError as e:
        print(f"Avertissement récupération souscriptions : {e.response['Error']['Message']}")

    # 2. Suppression du topic SNS
    try:
        sns_client.delete_topic(TopicArn=topic_arn)
        print(f"Topic SNS '{TOPIC_NAME}' supprimé avec succès.")
    except ClientError as e:
        print(f"Erreur suppression topic SNS : {e.response['Error']['Message']}")

    # 3. Suppression du fichier de configuration local
    if os.path.exists(CONFIG_FILE):
        os.remove(CONFIG_FILE)
        print(f"Fichier local '{CONFIG_FILE}' supprimé.")
else:
    print("Suppression désactivée. Passez CONFIRM_DELETE = True pour nettoyer les ressources SNS.")

Suppression désactivée. Passez CONFIRM_DELETE = True pour nettoyer les ressources SNS.
